# **HR Analytics EDA**

Цель анализа — понять, какие признаки кандидатов связаны с target = 1.

## Блок 1. Импорт библиотек

In [ ]:
import pandas as pd

: 

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
#чтобы Colab показывал больше колонок и строк, а не обрезал таблицы.

## Блок 2. Загрузка данных

In [ ]:
! gdown --id '1dXp6Gykzu7fZF0wa1N70Cl-LhdZpbfsw'

In [ ]:
! unzip /content/hr-analytics.zip

In [ ]:
train = pd.read_csv('/content/aug_train.csv')

In [ ]:
test = pd.read_csv('/content/aug_test.csv')

In [ ]:
sample_submission = pd.read_csv('/content/sample_submission.csv')

## Блок 3. Проверка размеров таблиц

**Что проверяем**:
- сколько строк и колонок в каждом файле
- есть ли target в train
- насколько test меньше train
- совпадает ли sample_submission по размеру с test

In [ ]:
print('train', train.shape)

In [ ]:
print('test', test.shape)

In [ ]:
print('sample_submission', sample_submission.shape)

### Описание файлов

В датасете есть три таблицы:

- `aug_train.csv` — основная таблица для анализа. Содержит признаки кандидатов и целевую переменную `target`.
- `aug_test.csv` — тестовая таблица. Содержит признаки кандидатов, но не содержит `target`.
- `sample_submission.csv` — шаблон ответа для ML-соревнования.

Основной EDA проводится на `aug_train.csv`, потому что только в этой таблице известна целевая переменная.

##Блок 5. Проверка колонок и первых строк

In [ ]:
train.head()

In [ ]:
print(train.columns.tolist())

## Блок 6. Описание колонок

Каждая строка в `train` соответствует одному кандидату.

- `enrollee_id` — идентификатор кандидата
- `city` — город кандидата
- `city_development_index` — индекс развития города
- `gender` — пол
- `relevent_experience` — наличие релевантного опыта
- `enrolled_university` — статус обучения в университете
- `education_level` — уровень образования
- `major_discipline` — основная специальность
- `experience` — опыт работы
- `company_size` — размер текущей или последней компании
- `company_type` — тип компании
- `last_new_job` — сколько лет назад кандидат менял работу
- `training_hours` — количество часов обучения
- `target` — целевая переменная

## Блок 7. Первичный осмотр train

In [ ]:
train.info() #Проверяем кол-во пустых значений во всех полях

In [ ]:
train.describe()
# std - стандартное отклонение, мера разброса или дисперсии данных относительно среднего значения.
#Большоре значение в std обозначает что точки данных сильно разбросаны от среднего

In [ ]:
train.describe(include = "object")

train.describe(include = "object")
- count (Количество): Общее количество непустых значений в этом категориальном столбце.
- unique (Уникальные): Количество уникальных категорий или значений в столбце.
- top (Самое частое): Самое часто встречающееся значение (категория) в столбце.
- freq (Частота): Количество раз, которое top значение встречается в столбце.

In [ ]:
train[['city', 'gender']].nunique()

- unique() возвращает массив (список) всех уникальных значений в Series.
- nunique() возвращает количество уникальных значений (целое число) в Series.

## Блок 8. Проверка дублей

In [ ]:
train.duplicated().sum()

In [ ]:
train.duplicated()

*   `train.duplicated()` ищет *идентичные копии* строк.
*   `df.isnull().all(axis=1)` ищет строки, состоящие *только из пустых значений*.

In [ ]:
train['enrollee_id'].duplicated().sum()

- если enrollee_id уникален, значит одна строка = один кандидат

## Блок 9. Выводы превичного анализа датасета
В `train` каждая строка соответствует одному кандидату.  
Ключевой идентификатор кандидата — `enrollee_id`.

## Блок 10. Анализ пропусков

Зачем:
Перед тем как сравнивать группы кандидатов, надо понять:

1. В каких колонках есть пропуски.
2. Насколько много пропусков.
3. Можно ли доверять признакам.
4. Не является ли сам факт пропуска важным сигналом.

In [ ]:
missing = (
    train.isna() #проверяет каждую ячейку на пропуск
    .sum()  #считает количество пропусков по каждой колонке
)
missing

In [ ]:
missing = (
    train.isna() #проверяет каждую ячейку на пропуск
    .sum()  #считает количество пропусков по каждой колонке
    .reset_index() #превращает результат в датафрейм
    .rename(columns={'index': 'column', 0: 'missing_count'}) #переименовывает колонки
)
missing['missing_share'] = missing['missing_count'] / len(train)
missing = missing.sort_values('missing_share', ascending=False)
missing["missing_share_pct"] = (missing["missing_share"] * 100).round(2)
missing
missing

## Объяснялка


- train.isna():

  Что делает: Этот метод проходится по каждой ячейке DataFrame train и проверяет, является ли она пропущенным значением (NaN, None).
  Результат: Возвращает новый DataFrame такой же формы, как train, но заполненный булевыми значениями True (если значение пропущено) или False (если значение присутствует).
- .sum():

  Что делает: После train.isna(), который дал нам DataFrame из True/False, метод .sum() суммирует эти булевы значения по столбцам. В Python True интерпретируется как 1, а False как 0.
  Результат: Возвращает Series, где индексом является название столбца, а значением — общее количество пропущенных значений в этом столбце.
- .reset_index():

  Что делает: Метод .sum() возвращает Series. Чтобы работать с ним как с полноценной таблицей, мы используем .reset_index(). Он преобразует Series в DataFrame, где старый индекс (column в нашем случае) становится обычным столбцом.
  Результат: DataFrame с двумя столбцами: один для названий колонок и один для количества пропусков.
- .rename(columns={'index': 'column', 0: 'missing_count'}):

  Что делает: После reset_index() столбцы обычно получают имена 'index' и '0' (или другое число). Этот метод переименовывает эти столбцы в более понятные 'column' и 'missing_count'.
  Результат: Тот же DataFrame, но с более осмысленными названиями столбцов.
- missing['missing_share'] = missing['missing_count'] / len(train):

  Что делает: Здесь мы вычисляем долю пропущенных значений для каждого столбца. Мы делим количество пропусков в каждом столбце (missing_count) на общее количество строк в train (len(train)).
  Результат: К DataFrame missing добавляется новый столбец 'missing_shape' с долей пропусков.
- missing = missing.sort_value- ('missing_shape', ascending=False):

  Что делает: Сортирует DataFrame missing по убыванию доли пропущенных значений (missing_shape), чтобы видеть столбцы с наибольшим количеством пропусков первыми.
  Результат: Отсортированный DataFrame missing.
- missing:

  Что делает: Просто выводит содержимое получившегося DataFrame missing.


## График

In [ ]:
import matplotlib.pyplot as plt #Как сделать темную тему для графиков
plt.style.use('dark_background')

In [ ]:
missing_nonzero = missing[missing["missing_count"] > 0] #новый DataFrame под названием missing_nonzero

plt.figure(figsize=(10, 5)) #создает окно для графика для Matplotlib.figsize=(10, 5) устанавливает размер фигуры в дюймах: 10 дюймов в ширину и 5 дюймов в высоту.
plt.bar(missing_nonzero["column"], missing_nonzero["missing_share_pct"]) #создания столбчатой диаграммы (bar chart)
#missing_nonzero["column"] - Ось Х
#missing_nonzero["missing_share_pct"] - Ось y
plt.xticks(rotation=45, ha="right") #Настраивает метки на оси X (названия колонок).
plt.title("Доля пропусков по колонкам") #Устанавливает заголовок для всего графика
plt.ylabel("Доля пропусков, %") #Устанавливает подпись для оси Y,
plt.show() #Отображает созданный график

1. missing_nonzero = missing

  [missing["missing_count"] > 0]:

  Эта строка создает новый DataFrame под названием missing_nonzero.
  Она фильтрует ваш уже существующий DataFrame missing (который содержит информацию обо всех пропусках).
  Фильтрация происходит по условию: missing["missing_count"] > 0. Это означает, что в missing_nonzero будут включены только те строки (то есть колонки из исходного DataFrame train), где количество пропущенных значений (missing_count) больше нуля. Таким образом, на графике будут показаны только те колонки, в которых есть пропуски.

2. plt.figure(figsize=(10, 5)):

  Эта строка создает новую фигуру (окно для графика) для Matplotlib.
  figsize=(10, 5) устанавливает размер фигуры в дюймах: 10 дюймов в ширину и 5 дюймов в высоту. Это помогает контролировать размер и пропорции вашего графика.

3. plt.bar(missing_nonzero["column"], missing_nonzero["missing_share_pct"]):

  Это основная команда для создания столбчатой диаграммы (bar chart).
  missing_nonzero["column"] предоставляет значения для оси X (названия колонок, в которых есть пропуски).
  missing_nonzero["missing_share_pct"] предоставляет значения для оси Y (процент пропущенных значений для каждой колонки).

4. plt.xticks(rotation=45, ha="right"):

  Настраивает метки на оси X (названия колонок).
  rotation=45 поворачивает текст меток на 45 градусов, чтобы длинные названия не накладывались друг на друга.
  ha="right" (horizontal alignment) выравнивает текст меток по правому краю относительно точки поворота, что улучшает читаемость.

5. plt.title("Доля пропусков по колонкам"):

  Устанавливает заголовок для всего графика.

6. plt.ylabel("Доля пропусков, %"):

  Устанавливает подпись для оси Y, поясняя, что она отображает процент пропущенных значений.

7. plt.show():

  Отображает созданный график. Если вы не используете

### Выводы по пропускам

В датасете есть признаки с заметной долей пропусков. Больше всего пропусков ожидается в признаках, связанных с текущей или прошлой компанией кандидата: `company_type` и `company_size`.

Также пропуски есть в `gender`, `major_discipline`, `education_level`, `enrolled_university`, `experience` и `last_new_job`.

На данном этапе пропуски не удаляем, потому что сам факт отсутствия информации может быть связан с `target`.

## Блок 11. Анализ целевой переменной target

`target` — главный признак для анализа.  
Сначала посмотрим, какая доля кандидатов имеет `target = 1`.

In [ ]:
train['target'].value_counts()

In [ ]:
train['target'].value_counts(normalize=True)

## Объяснялка

параметр normalize=True, метод возвращает относительную частоту (долю или процент) каждого уникального значения

## Блок 11.2

In [ ]:
target_rate = train["target"].mean()
target_rate

In [ ]:
target_summary = train["target"].value_counts().reset_index()
#подсчитывает количество вхождений каждого уникального значения в столбце target
#reset_index() - метод преобразует полученный объект Series (у которого значения target были индексом, а частоты — значениями) в DataFrame
target_summary.columns = ["target", "count"] #Эта строка явно переименовывает колонки DataFrame target_summary
target_summary["share_pct"] = (target_summary["count"] / len(train) * 100).round(2)
target_summary

In [ ]:
plt.figure(figsize=(5, 4)) #Задаем размеры графика
plt.bar(target_summary["target"].astype(str), target_summary["count"]) #
plt.title("Распределение target")
plt.xlabel("target")
plt.ylabel("Количество кандидатов")

# Автоматически подбираем отступы, чтобы подписи не обрезались
plt.tight_layout()

# Сохраняем график в папку images, расположенную рядом с папкой notebook
plt.savefig(
    "../images/target_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Вывод по target

Доля кандидатов с `target = 1` составляет примерно 25%.  
Это значит, что около каждого четвертого кандидата в обучающей выборке потенциально заинтересован в смене работы.

Дальше эту долю будем использовать как базовый уровень для сравнения разных групп кандидатов.

## Блок 12. Анализ категориальных признаков

На этом шаге будем смотреть, как доля `target = 1` отличается между группами кандидатов.

Для каждой категории считаем:

- `candidates` — количество кандидатов в группе;
- `target_rate` — долю кандидатов с `target = 1`;
- `candidate_share_pct` — долю группы от всего датасета;
- `target_rate_diff_pp` — отличие от среднего уровня `target_rate` в процентных пунктах.

In [ ]:
def categorical_target_summary(df, column):
    result = (
        df.groupby(column, dropna=False)
        .agg(
            candidates=("enrollee_id", "count"),
            target_rate=("target", "mean")
        )
        .reset_index()
    )

    result["candidate_share_pct"] = (result["candidates"] / len(df) * 100).round(2)
    result["target_rate_pct"] = (result["target_rate"] * 100).round(2)
    result["target_rate_diff_pp"] = ((result["target_rate"] - df["target"].mean()) * 100).round(2)

    result = result.sort_values("target_rate", ascending=False)

    return result

1. def categorical_target_summary(df, column): Мы создаем функцию с двумя входами:
df — датафрейм, например train
column — название колонки, по которой хотим группировать
2. result = (
    df.groupby(column, dropna=False):
Мы берем датафрейм df и группируем его по выбранной колонке.
3. .agg(
    candidates=("enrollee_id", "count"),
    target_rate=("target", "mean"))
candidates - сколько строк
target_rate - среднее значение target внутри группы
4. .reset_index() - После groupby выбранная колонка становится индексом таблицы.
reset_index() возвращает ее обратно в обычную колонку
5. result["target_rate_pct"] = (result["target_rate"] * 100).round(2) - переводим target из доли в проценты

In [ ]:
train

## Блок 12.1. Релевантный опыт

Проверим, отличается ли доля кандидатов с `target = 1` между группами с релевантным опытом и без него.

In [ ]:
exp_summary = categorical_target_summary(train, "relevent_experience")
exp_summary

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    exp_summary["relevent_experience"].astype(str),
    exp_summary["target_rate_pct"]
)

plt.axhline(y=target_rate * 100, linestyle="--")

plt.title("Доля target = 1 по наличию релевантного опыта")
plt.xlabel("Есть ли релевантный опыт")
plt.ylabel("Доля target = 1, %")
plt.xticks(rotation=15, ha="right")

plt.show()

### Вывод

Кандидаты без релевантного опыта имеют более высокую долю `target = 1`, чем кандидаты с релевантным опытом.

Это может означать, что кандидаты без релевантного опыта чаще рассматривают обучение как способ перейти на новую работу.

## Блок 12.2. Сводный анализ категориальных признаков

Посмотрим, как `target_rate` отличается по основным категориальным признакам.

In [ ]:
categorical_cols = [
    "gender",
    "relevent_experience",
    "enrolled_university",
    "education_level",
    "major_discipline",
    "experience",
    "company_size",
    "company_type",
    "last_new_job"
]

for col in categorical_cols:
    print("\n" + "=" * 80)
    print(col)
    display(categorical_target_summary(train, col))

## Блок 12.3. Основные наблюдения по категориальным признакам

По результатам анализа категориальных признаков можно выделить несколько предварительных наблюдений:

1. Кандидаты без релевантного опыта имеют заметно более высокий `target_rate`: 33.84% против 21.47% у кандидатов с релевантным опытом. Это на 8.91 процентного пункта выше среднего уровня по датасету.

2. Статус обучения в университете связан с различиями в `target_rate`. У кандидатов, проходящих full-time course, доля `target = 1` составляет 38.09%, что на 13.15 процентного пункта выше среднего. У кандидатов без текущего обучения (`no_enrollment`) показатель ниже среднего — 21.14%.

3. Опыт работы показывает выраженный паттерн: кандидаты с небольшим опытом чаще имеют `target = 1`. Например, у группы `<1` год `target_rate` составляет 45.40%, а у группы `1` год — 42.44%. У кандидатов с опытом `>20` лет показатель существенно ниже — 15.31%.

4. Пропуски в данных о компании выглядят информативными. У кандидатов с пропущенным `company_size` доля `target = 1` составляет 40.59%, а у кандидатов с пропущенным `company_type` — 38.83%. Это выше среднего уровня по датасету.

5. По признаку `last_new_job` видно, что кандидаты, которые никогда не меняли работу (`never`), имеют более высокий `target_rate` — 30.14%. Кандидаты, которые меняли работу более 4 лет назад (`>4`), имеют более низкий показатель — 18.24%.

6. По `education_level` самый высокий `target_rate` наблюдается у группы `Graduate` — 27.98%. У групп `Phd` и `Primary School` показатель ниже среднего, но эти группы занимают небольшую долю датасета, поэтому выводы по ним нужно делать осторожно.

На данном этапе речь идет о связи признаков с `target`, а не о доказанной причинно-следственной зависимости.

## Блок 13. Анализ числовых показателей

Зачем:сравнить числовые признаки между target = 0 и target = 1. В этом датасете основные числовые признаки — city_development_index и training_hours

In [ ]:
numeric_cols = ["city_development_index", "training_hours"]

train.groupby("target")[numeric_cols].agg(["count", "mean", "median", "min", "max"])

**Гистограмма по city_development_index:**

In [ ]:
plt.figure(figsize=(7, 4))

plt.hist(train.loc[train["target"] == 0, "city_development_index"], alpha=0.5, label="target = 0")
plt.hist(train.loc[train["target"] == 1, "city_development_index"], alpha=0.5, label="target = 1")

plt.title("Распределение city_development_index по target")
plt.xlabel("city_development_index")
plt.ylabel("Количество кандидатов")
plt.legend()

plt.show()

## Блок 14. Группировка числовых признаков в интервалы

Числовые признаки не всегда удобно анализировать по отдельным значениям.  
Поэтому разобьем их на группы:

- `city_development_index` — на смысловые интервалы: low, medium, high, very_high;
- `training_hours` — на 4 группы по квартилям.

После этого сравним `target_rate` внутри полученных групп.

**14.1. Разбиваем city_development_index на группы**:

In [ ]:
train["city_development_group"] = pd.cut(
    train["city_development_index"],
    bins=[0, 0.65, 0.8, 0.9, 1.0],
    labels=["low", "medium", "high", "very_high"]
)

- pd.cut — функция pandas для разделения непрерывной переменной на интервалы (бины).
- train["city_development_index"], фиксируем числовую колонку "city_development_index"
- bins=[0, 0.65, 0.8, 0.9, 1.0], задает границы интервалов
- labels=["low", "medium", "high", "very_high"] дает каждому интервалу название:

0–0.65       → low
0.65–0.8     → medium
0.8–0.9      → high
0.9–1.0      → very_high

**14.2. Проверяем, что новая колонка создалась**

In [ ]:
train[['city_development_index', 'city_development_group']]

**14.3. Считаем target_rate по группам city development**

In [ ]:
city_dev_summary = categorical_target_summary(train, "city_development_group")
city_dev_summary

candidates — количество кандидатов в группе
target_rate — долю target = 1
candidate_share_pct — долю группы от всего датасета
target_rate_pct — target_rate в процентах
target_rate_diff_pp — отличие от среднего target_rate

14.4. График по группам city_development_index

In [ ]:
plt.figure(figsize = (7, 4))

plt.bar(
    city_dev_summary["city_development_group"].astype(str),
    city_dev_summary["target_rate_pct"]
)
plt.axhline(y = target_rate * 100, linestyle = "--")

plt.title("Доля target = 1 по city_development_group")
plt.xlabel("city_development_group")
plt.ylabel('Доля target = 1, %')

plt.show()

14.5. Разбиваем training_hours на 4 группы

In [ ]:
train

In [ ]:
train["training_hours_group"] = pd.qcut(
    train["training_hours"],
    q=4,
    labels=["low", "medium", "high", "very_high"]
)

pd.qcut (Квантильная дискретизация)
Назначение: Разбивает числовые данные на группы так, чтобы в каждой группе было примерно одинаковое количество элементов.
Параметры:
x: Числовой столбец, который мы разбиваем (например, train["training_hours"]).
q:
q=4: Разбить на 4 группы (квартили). Каждая группа содержит ~25% данных.
Можно задать списком: q=[0, 0.25, 0.5, 0.75, 1] для тех же квартилей.
duplicates="drop": Если границы интервалов совпадают из-за повторяющихся значений, то лишние границы удаляются. Может уменьшить количество групп.
pd.cut (Дискретизация по интервалам)
Назначение: Разбивает числовые данные на интервалы равной ширины или по явно заданным границам.
Параметры (ключевые):
x: Числовой столбец, который мы разбиваем.
bins:
bins=4: Разбить на 4 интервала равной ширины.
Можно задать списком: bins=[0, 0.65, 0.8, 0.9, 1.0] для своих границ.
labels: Список текстовых меток для каждого интервала (например, ["low", "medium", "high", "very_high"]).
Главное отличие:
qcut: Делит так, чтобы количество данных в каждой группе было примерно равным.
cut: Делит так, чтобы ширина интервалов была равной (или по заданным вами границам).


In [ ]:
train["training_hours_group"].value_counts().sort_index()

1. train["training_hours_group"].value_counts():

    Что делает: Этот метод подсчитывает, сколько раз встречается каждое уникальное значение (каждая группа) в колонке training_hours_group.

2. sort_index():

    Что делает: После того как value_counts() выдал результат (который является Series), .sort_index() берет этот Series и сортирует его по индексу - в нашем случае по алфавиту .



14.6. Проверяем группы training_hours

In [ ]:
train[["training_hours", "training_hours_group"]].head(10)

In [ ]:
train['training_hours_group'].value_counts()

14.7. Считаем target_rate по группам training_hours

In [ ]:
training_hours_summary = categorical_target_summary(train, "training_hours_group")
training_hours_summary

14.8. График по training_hours_group

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    training_hours_summary["training_hours_group"].astype(str),
    training_hours_summary["target_rate_pct"]
)

plt.axhline(y=target_rate * 100, linestyle="--")

plt.title("Доля target = 1 по группам training_hours")
plt.xlabel("Группа training_hours")
plt.ylabel("Доля target = 1, %")
plt.xticks(rotation=30, ha="right")

plt.show()

### Вывод по группировке числовых признаков

После разбиения числовых признаков на интервалы стало проще сравнивать группы между собой.

`city_development_index` был преобразован в смысловые группы: low, medium, high и very_high.  
`training_hours` был разбит на 4 группы по квартилям.

Дальше по таблицам `city_dev_summary` и `training_hours_summary` можно сравнить, в каких интервалах доля `target = 1` выше или ниже среднего уровня по датасету.

## Блок 15. Анализ городов

Проверим, есть ли города с заметно высоким или низким `target_rate`.

Так как городов много, будем делать выводы только по городам, где есть достаточное количество кандидатов.  
Для маленьких групп `target_rate` может быть случайно высоким или низким.

In [ ]:
city_summary = (
    train.groupby("city")
    .agg(
        candidates=("enrollee_id", "count"),
        target_rate=("target", "mean"),
        avg_city_development_index=("city_development_index", "mean")
    )
    .reset_index()
)

city_summary["target_rate_pct"] = (city_summary["target_rate"] * 100).round(2)
city_summary["candidate_share_pct"] = (city_summary["candidates"] / len(train) * 100).round(2)
city_summary["target_rate_diff_pp"] = ((city_summary["target_rate"] - target_rate) * 100).round(2)

city_summary = city_summary.sort_values("target_rate", ascending=False)

city_summary.head(10)

In [ ]:
big_cities = city_summary[city_summary["candidates"] >= 100]

In [ ]:
big_cities

In [ ]:
big_cities.sort_values("target_rate", ascending=False).head(10)

In [ ]:
big_cities.sort_values("target_rate", ascending=True).head(10)

График топ-10 городов по target_rate:

In [ ]:
top_cities = big_cities.sort_values("target_rate", ascending=False).head(10)

plt.figure(figsize=(9, 5))

plt.barh( #barh — горизонтальный bar chart.Он лучше подходит, когда названия категорий длинные или категорий много.
    top_cities["city"],
    top_cities["target_rate_pct"]
)

plt.axvline(x=target_rate * 100, linestyle="--")

plt.title("Топ-10 городов по доле target = 1")
plt.xlabel("Доля target = 1, %")
plt.ylabel("Город")

plt.show()

### Вывод по городам

Для анализа городов использовался фильтр `candidates >= 100`, чтобы не делать выводы по слишком маленьким группам.

Города с высоким или низким `target_rate` можно рассматривать как отдельные сегменты, но выводы по ним нужно делать осторожно: различия могут быть связаны не только с самим городом, но и с составом кандидатов внутри города.

## Блок 16. Сводные таблицы по сочетаниям признаков

Посмотрим не только отдельные признаки, но и их сочетания.

Например:
- уровень образования + релевантный опыт;
- уровень развития города + статус обучения.

Такие таблицы помогают увидеть, усиливаются ли различия в `target_rate` при пересечении признаков.

In [ ]:
pivot_edu_exp = train.pivot_table(
    index="education_level",
    columns="relevent_experience",
    values="target",
    aggfunc="mean"
)

pivot_edu_exp_pct = (pivot_edu_exp * 100).round(2)
pivot_edu_exp_pct

- train.pivot_table(...): Это первая и основная операция. Вызывается метод pivot_table для DataFrame train.
- index="education_level": Pandas берет все уникальные значения из столбца education_level (например, 'Graduate', 'Masters', 'Phd' и т.д.) и делает их индексами (строками) новой таблицы.
- columns="relevent_experience": Pandas берет все уникальные значения из столбца relevent_experience ('Has relevent experience', 'No relevent experience') и делает их названиями новых столбцов в новой таблице.
- values="target": В качестве значений для заполнения ячеек таблицы используется столбец target.
- aggfunc="mean": Это функция агрегации. Для каждой комбинации education_level и relevent_experience (например, 'Graduate' и 'Has relevent experience') pandas вычисляет среднее значение target. Поскольку target содержит 0 и 1, среднее значение будет долей target = 1 в этой группе.
- Результат этой операции: Создается DataFrame pivot_edu_exp, где строки — уровни образования, столбцы — наличие опыта, а значения — средняя доля target = 1 для каждой комбинации.

Вариант heatmap на matplotlib:

In [ ]:
plt.figure(figsize=(7, 4)) #создали окно под heatmap, задали размер окна (7 в ширину 4 в высоту)

plt.imshow(pivot_edu_exp_pct, aspect="auto") #Отображает данные в виде изображения, где значения в ячейках 'pivot_edu_exp_pct' кодируются цветом. Это основной метод для создания heatmap

plt.colorbar(label="Доля target = 1, %")
plt.xticks(range(len(pivot_edu_exp_pct.columns)), pivot_edu_exp_pct.columns, rotation=30, ha="right") #находим количество колонок - эти колонки становятся поззициями на Ох
plt.yticks(range(len(pivot_edu_exp_pct.index)), pivot_edu_exp_pct.index)

plt.title("Target rate: education_level × relevent_experience")
plt.xlabel("Релевантный опыт")
plt.ylabel("Уровень образования")

plt.show()

Вариант heatmap на matplotlib:

In [ ]:
plt.figure(figsize=(8, 5)) #задаем размер области

sns.heatmap( #Функция seaborn для создания heatmap
    pivot_edu_exp_pct, #DataFrame, который будет отображен в виде тепловой карты. Значения из этого DataFrame будут определять цвет каждой ячейки
    annot=True, #Если True, значения данных будут отображаться в каждой ячейке тепловой карты
    fmt=".2f", #Используется вместе с annot=True. Определяет формат отображения чисел в ячейках. ".2f" означает, что числа будут округлены до двух знаков после запятой
    cmap="YlGnBu", #Задает цветовую палитру (цветовую карту) для тепловой карты. "YlGnBu" — это одна из встроенных палитр seaborn, где цвета переходят от желтого (Yl) к зеленому (Gn) и затем к синему (Bu)
    cbar_kws={"label": "Доля target = 1, %"} #Это словарь с дополнительными аргументами, которые будут переданы для настройки цветовой шкалы (colorbar). Здесь мы устанавливаем подпись для цветовой шкалы: "Доля target = 1, %".

)

plt.title("Target rate: education_level × relevent_experience")
plt.xlabel("Релевантный опыт")
plt.ylabel("Уровень образования")

plt.show()

## Блок 17. Phik-корреляция

Обычная корреляция Pearson подходит в основном для числовых признаков.  
В этом датасете много категориальных признаков, поэтому дополнительно посмотрим Phik-корреляцию.

Phik можно воспринимать как способ оценить силу связи между признаками разных типов: числовыми и категориальными.

In [ ]:
!pip install phik -q

In [ ]:
import phik

In [ ]:
phik_df = train.copy() #Подготавливаем копию данных, чтобы не портить основной train

In [ ]:
phik_cols = [ #Выберем признаки (Создаем список колонок, которые хотим включить в Phik-анализ):
    "city_development_index",
    "training_hours",
    "gender",
    "relevent_experience",
    "enrolled_university",
    "education_level",
    "major_discipline",
    "experience",
    "company_size",
    "company_type",
    "last_new_job",
    "target"
]

phik_df = train.copy()[phik_cols] #Оставляем в phik_df только выбранные колонки
#copy() - тк мы создали независимый датафрейм, который можно безопасно менять

In [ ]:
phik_matrix = phik_df.phik_matrix(
    interval_cols=["city_development_index", "training_hours"] #interval_cols — это список колонок, которые мы явно сообщаем библиотеке как непрерывные числовые признаки.
)

phik_matrix["target"].sort_values(ascending=False)

Phik по умолчанию нормально работает с категориальными колонками как с категориями.
А вот числовые непрерывные признаки ему лучше явно указать, чтобы он обрабатывал их как интервальные числовые переменные, а не как набор отдельных категорий.

Heatmap по Phik:

In [ ]:
plt.figure(figsize=(10, 7)) #Создаем область графика размером 10 на 7.

plt.imshow(phik_matrix, aspect="auto")

plt.colorbar(label="Phik correlation")
plt.xticks(range(len(phik_matrix.columns)), phik_matrix.columns, rotation=90)
plt.yticks(range(len(phik_matrix.index)), phik_matrix.index)

plt.title("Phik-корреляция между признаками")

plt.show()

In [ ]:
plt.figure(figsize=(10, 7)) #Создаем область графика размером 10 на 7.

sns.heatmap(
    phik_matrix, #таблица значений, которую визуализируем
    annot=True, #показывает числа внутри ячеек
    fmt=".2f", #округляет числа до двух знаков после запятой
    cmap="YlGnBu" #задает цветовую шкалу.
)

plt.title("Phik-корреляция между признаками") #Добавляет заголовок

# Автоматически подбираем отступы, чтобы подписи не обрезались
plt.tight_layout()

# Сохраняем график в папку images, расположенную рядом с папкой notebook
plt.savefig(
    "../images/phik_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show() #Показывает график.

### Вывод по Phik

Phik-корреляция помогает оценить силу связи между `target` и признаками разных типов.

Этот блок не заменяет анализ через `groupby`, но помогает быстро увидеть, какие признаки потенциально сильнее связаны с целевой переменной.

## Блок 17.1. Подробный анализ зависимостей после Phik

Phik-матрица показывает, между какими признаками есть связь, но не объясняет форму этой связи.

Поэтому после Phik выбираем несколько интересных пар признаков и смотрим их подробнее:

1. `city_development_index` и `target` — числовой признак + бинарный target.
2. `relevent_experience` и `last_new_job` — категориальный признак + категориальный признак.
3. `relevent_experience`, `last_new_job` и `target` — смотрим, как сочетание двух признаков связано с `target_rate`.

Цель блока — не просто увидеть наличие связи, а понять, как именно она проявляется в данных.

17.1.1. Таблица признаков, связанных с target по Phik

In [ ]:
# Берем из Phik-матрицы только колонку "target"
# В этой колонке лежит сила связи каждого признака с target
target_phik = phik_matrix["target"]

# Убираем строку "target", потому что target с самим собой всегда имеет связь 1.0
#(Когда вы вызываете метод .drop() на объекте pandas.Series, pandas по умолчанию пытается удалить элемент из индекса (строки) этого Series)
target_phik = target_phik.drop("target")

# Сортируем признаки от самой сильной связи с target к самой слабой
target_phik = target_phik.sort_values(ascending=False)

# Превращаем Series в обычный датафрейм
target_phik = target_phik.reset_index()

# Переименовываем колонки, чтобы таблица читалась понятно
target_phik.columns = ["feature", "phik_with_target"]

# Округляем значения Phik до двух знаков после запятой
target_phik["phik_with_target"] = target_phik["phik_with_target"].round(2)

# Выводим итоговую таблицу
target_phik

17.1.2. Связь city_development_index и target

Phik показал, что city_development_index заметно связан с target. Теперь смотрим форму связи.

In [ ]:
# Делим city_development_index на 4 группы по квартилям
# qcut делает группы так, чтобы в каждой было примерно одинаковое количество кандидатов
_, city_bins = pd.qcut(
    train["city_development_index"], # числовая колонка, которую делим на группы
    q=4,                             # делим на 4 квартиля
    retbins=True,                    # дополнительно возвращаем границы интервалов, Этот параметр указывает функции pd.qcut, что нужно вернуть фактические границы интервалов (bins), которые были использованы для разделения данных.
    duplicates="drop"                # если границы повторяются, pandas не сломает код, а уберет дубли, Если duplicates="drop", pandas автоматически удаляет повторяющиеся границы, чтобы избежать ошибки. Это приводит к тому, что количество результирующих интервалов может быть меньше, чем указано в q. Если не указать "drop" (или "raise"), pd.qcut выбросит ошибку ValueError в такой ситуации.
)

# Создаем базовые названия групп
city_base_labels = ["low", "medium", "high", "very_high"]

# Создаем подписи групп с фактическими диапазонами значений
city_labels = [
    f"{city_base_labels[i]} ({city_bins[i]:.3f}–{city_bins[i + 1]:.3f})" # форматируем границы до 3 знаков
    for i in range(len(city_bins) - 1)                                  # создаем подпись для каждого интервала
]

# Создаем новую колонку с группами city_development_index
train["city_development_qgroup"] = pd.qcut(
    train["city_development_index"], # колонка, которую разбиваем
    q=4,                             # снова делим на 4 группы
    labels=city_labels,              # используем понятные названия с диапазонами
    duplicates="drop"                # защита от повторяющихся границ
)

# Группируем данные по новым группам city_development_qgroup
city_dev_q_summary = (
    train.groupby("city_development_qgroup", observed=False) # группировка по квартильным группам
    .agg(
        candidates=("enrollee_id", "count"),                 # считаем количество кандидатов в группе
        target_rate=("target", "mean")                       # считаем долю target = 1 внутри группы
    )
    .reset_index()                                           # возвращаем группу из индекса в обычную колонку
)

# Переводим target_rate из доли в проценты
city_dev_q_summary["target_rate_pct"] = (city_dev_q_summary["target_rate"] * 100).round(2)

# Считаем отличие target_rate группы от среднего target_rate по всему датасету
city_dev_q_summary["target_rate_diff_pp"] = (
    (city_dev_q_summary["target_rate"] - target_rate) * 100
).round(2)

# Выводим таблицу
city_dev_q_summary

График по Связи city_development_index и target:

In [ ]:
# Создаем область графика
plt.figure(figsize=(9, 4))

# Строим line plot: по X группы city_development_index, по Y target_rate в процентах
plt.plot(
    city_dev_q_summary["city_development_qgroup"].astype(str), # ось X: группы city_development_index
    city_dev_q_summary["target_rate_pct"],                     # ось Y: доля target = 1
    marker="o"                                                 # добавляем точки на линии
)

# Добавляем горизонтальную линию среднего target_rate по всему датасету
plt.axhline(y=target_rate * 100, linestyle="--")

# Добавляем заголовок графика
plt.title("Target rate по квартилям city_development_index")

# Подписываем ось X
plt.xlabel("Группа city_development_index")

# Подписываем ось Y
plt.ylabel("Доля target = 1, %")

# Поворачиваем подписи на оси X, чтобы они не налезали друг на друга
plt.xticks(rotation=30, ha="right")

# Показываем график

# Автоматически подбираем отступы, чтобы подписи не обрезались
plt.tight_layout()

# Сохраняем график в папку images, расположенную рядом с папкой notebook
plt.savefig(
    "../images/city_development_target_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.plot(): Для трендов и последовательностей (как Y меняется по отношению к X). Соединяет точки линией.
plt.hist(): Для распределения одного числового признака (сколько значений попадает в каждый интервал).
plt.bar(): Для сравнения значений между дискретными категориями (высота столбца показывает значение).
sns.heatmap(): Для визуализации матриц и сводных таблиц, где цвет ячейки кодирует значение, показывая взаимосвязь двух переменных с третьим показателем.

### Вывод по `city_development_index` и `target`

Phik показал заметную связь между `city_development_index` и `target`.

После разбиения `city_development_index` на квартильные группы видно, как меняется `target_rate` при переходе от низких значений индекса к высоким.

Если в группах с низким `city_development_index` доля `target = 1` выше, это означает, что кандидаты из менее развитых городов чаще заинтересованы в смене работы.

## 17.1.3. Связь relevent_experience и last_new_job

Здесь обе переменные категориальные. Поэтому вопрос “линейная или нелинейная связь” тут не подходит. Нужно смотреть распределение категорий.

In [ ]:
# Строим таблицу сопряженности между relevent_experience и last_new_job
rel_exp_last_job = pd.crosstab(
    train["relevent_experience"], # строки таблицы
    train["last_new_job"],        # столбцы таблицы
    normalize="index"             # считаем доли внутри каждой строки
)

# Переводим доли в проценты
rel_exp_last_job = rel_exp_last_job * 100

# Округляем значения до двух знаков после запятой
rel_exp_last_job = rel_exp_last_job.round(2)

# Выводим таблицу
rel_exp_last_job

Heatmap по связи relevent_experience и last_new_job:

In [ ]:
# Создаем область графика
plt.figure(figsize=(8, 4))

# Строим heatmap по таблице сопряженности
sns.heatmap(
    rel_exp_last_job,                         # таблица, которую визуализируем
    annot=True,                               # показываем числа внутри ячеек
    fmt=".2f",                                # формат чисел: два знака после запятой
    cmap="YlGnBu",                            # цветовая шкала
    cbar_kws={"label": "Доля внутри группы, %"} # подпись цветовой шкалы
)

# Добавляем заголовок
plt.title("Распределение last_new_job внутри relevent_experience")

# Подписываем ось X
plt.xlabel("last_new_job")

# Подписываем ось Y
plt.ylabel("relevent_experience")

# Показываем график
plt.show()

### Вывод по `relevent_experience` и `last_new_job`

Phik показал заметную связь между `relevent_experience` и `last_new_job`.

Через `crosstab` видно, как распределяются категории `last_new_job` внутри групп с релевантным опытом и без него.

Так как обе переменные категориальные, здесь мы не говорим о линейной или нелинейной связи. Связь проявляется в различии распределений категорий.

17.1.4. Как сочетание relevent_experience и last_new_job связано с target

In [ ]:
# Создаем сводную таблицу
# В строках будет last_new_job
# В столбцах будет relevent_experience
# В ячейках будет средний target, то есть target_rate
pivot_rel_exp_last_job_target = train.pivot_table(
    index="last_new_job",             # строки сводной таблицы
    columns="relevent_experience",    # столбцы сводной таблицы
    values="target",                  # значение, которое агрегируем
    aggfunc="mean"                    # среднее target = доля target = 1
)

# Переводим значения из долей в проценты
pivot_rel_exp_last_job_target = pivot_rel_exp_last_job_target * 100

# Округляем значения до двух знаков после запятой
pivot_rel_exp_last_job_target = pivot_rel_exp_last_job_target.round(2)

# Выводим таблицу
pivot_rel_exp_last_job_target

Heatmap по связи relevent_experience и last_new_job:

In [ ]:
plt.figure(figsize=(8, 4))

sns.heatmap(
    pivot_rel_exp_last_job_target, #Задаем название таблицы
    annot=True, #Задаем параметр чтобы в каждой ечейке были числа
    fmt=".2f", #Для каждого числового значения в ячейках будет максимум два знака после запятой
    cmap = "YlGnBu", #Задаем цветовую палитру для ячеек - от желтого потом Зеленый потом Голубой
    cbar_kws = {'label' : 'Доля target = 1, %'} #ПОдписываем легенду для шкалы цвета
)
# Добавляем заголовок

plt.title("Target rate: last_new_job × relevent_experience")
# Подписываем ось X

plt.xlabel("relevent_experience")

# Подписываем ось Y

plt.ylabel("last_new_job")
# Показываем график


# Автоматически подбираем отступы, чтобы подписи не обрезались
plt.tight_layout()

# Сохраняем график в папку images, расположенную рядом с папкой notebook
plt.savefig(
    "../images/experience_last_job_target_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



### Вывод по пересечению `last_new_job` и `relevent_experience`

Сводная таблица показывает, как `target_rate` меняется на пересечении двух признаков: `last_new_job` и `relevent_experience`.

Такой анализ полезен после Phik, потому что Phik показывает наличие связи, а `pivot_table` помогает понять, как эта связь проявляется относительно `target`.

Если у кандидатов без релевантного опыта `target_rate` выше почти во всех группах `last_new_job`, это усиливает вывод о связи `relevent_experience` с интересом к смене работы.

## Блок 18. Итоговые выводы

По результатам EDA можно выделить несколько основных выводов:

1. Доля кандидатов с `target = 1` составляет примерно 25%.
2. В данных есть заметные пропуски в `company_type`, `company_size`, `gender` и `major_discipline`.
3. Кандидаты без релевантного опыта чаще имеют `target = 1`, чем кандидаты с релевантным опытом.
4. Кандидаты, проходящие full-time course, имеют `target_rate` выше среднего.
5. Кандидаты с небольшим опытом работы чаще имеют `target = 1`.
6. Пропуски в признаках о компании выглядят информативными и не должны автоматически удаляться.
7. Числовые признаки удобнее анализировать через интервалы: это делает выводы проще и ближе к BI-логике.
8. Анализ городов нужно делать осторожно, потому что размер групп сильно отличается.
9. Phik-корреляция может использоваться как дополнительный способ быстро оценить связь признаков с `target`.

Важно: EDA показывает связи между признаками и `target`, но не доказывает причинно-следственные зависимости.